# Text Preprocessing

## Objectives

- Clean raw text
- Build a reusable preprocessing pipeline
- Apply preprocessing to train and test data

In [1]:
import re
import string
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /usr/share/nltk_data...


True

In [2]:
import pandas as pd

train = pd.read_csv("/kaggle/input/datasets/julian3833/jigsaw-toxic-comment-classification-challenge/train.csv")
test = pd.read_csv("/kaggle/input/datasets/julian3833/jigsaw-toxic-comment-classification-challenge/test.csv")

target_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate"
]

In [3]:
stop_words = set(stopwords.words("english"))

lemmatizer = WordNetLemmatizer()

In [4]:
def preprocess_text(text):
    """
    Cleans a single comment.

    Steps:
    1. Lowercase
    2. Remove URLs
    3. Remove HTML
    4. Remove punctuation
    5. Remove numbers
    6. Remove extra spaces
    7. Remove stopwords
    8. Lemmatize words
    """

    text = str(text).lower()

    text = re.sub(r"http\S+|www\S+", "", text)

    text = re.sub(r"<.*?>", "", text)

    text = re.sub(r"\d+", "", text)

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()

    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]

    return " ".join(words)

In [5]:
sample = train.loc[0, "comment_text"]

print("Original:\n")
print(sample)

print("\n" + "="*80 + "\n")

print("Processed:\n")
print(preprocess_text(sample))

Original:

Explanation
Why the edits made under my username Hardcore Metallica Fan were reverted? They weren't vandalisms, just closure on some GAs after I voted at New York Dolls FAC. And please don't remove the template from the talk page since I'm retired now.89.205.38.27


Processed:

explanation edits made username hardcore metallica fan reverted werent vandalism closure gas voted new york doll fac please dont remove template talk page since im retired


In [6]:
train["clean_comment"] = train["comment_text"].apply(preprocess_text)

test["clean_comment"] = test["comment_text"].apply(preprocess_text)

In [10]:
train[
    ["comment_text", "clean_comment"]
].head()

,comment_text,clean_comment
0,Explanation\nWhy the edits made under my usern...,explanation edits made username hardcore metal...
1,D'aww! He matches this background colour I'm s...,daww match background colour im seemingly stuc...
2,"Hey man, I'm really not trying to edit war. It...",hey man im really trying edit war guy constant...
3,"""\nMore\nI can't make any real suggestions on ...",cant make real suggestion improvement wondered...
4,"You, sir, are my hero. Any chance you remember...",sir hero chance remember page thats


## Completed

- Converted text to lowercase
- Removed URLs
- Removed HTML tags
- Removed punctuation
- Removed numbers
- Removed extra spaces
- Removed stopwords
- Applied lemmatization

The cleaned text is now ready for TF-IDF vectorization and model training.

# TF-IDF Feature Engineering

## Objectives

- Split the data
- Convert text into numerical features
- Learn the TF-IDF vocabulary
- Save the fitted vectorizer

In [8]:
import pandas as pd
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

In [9]:
X = train["clean_comment"]

y = train[
    [
        "toxic",
        "severe_toxic",
        "obscene",
        "threat",
        "insult",
        "identity_hate",
    ]
]

In [12]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
)

In [13]:
tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1,2),
    min_df=3,
    max_df=0.9
)

In [14]:
X_train_tfidf = tfidf.fit_transform(X_train)
X_valid_tfidf = tfidf.transform(X_valid)

In [15]:
print("Training Shape :", X_train_tfidf.shape)
print("Validation Shape :", X_valid_tfidf.shape)

Training Shape : (127656, 30000)
Validation Shape : (31915, 30000)


In [16]:
feature_names = tfidf.get_feature_names_out()
feature_names[:25]

array(['aa', 'aap', 'aaron', 'ab', 'abandon', 'abandoned', 'abbas',
       'abbey', 'abbreviated', 'abbreviation', 'abc', 'abc news', 'abd',
       'abdul', 'abe', 'abide', 'ability', 'ability create',
       'ability edit', 'abkhazia', 'able', 'able contribute', 'able edit',
       'able find', 'able get'], dtype=object)

## Completed

- Cleaned the text
- Split the dataset
- Converted text into TF-IDF vectors
- Learned the vocabulary

The processed feature matrices (`X_train_tfidf`, `X_valid_tfidf`) and labels (`y_train`, `y_valid`) are now ready for training classical machine learning models.